# Web Scraping X (Twitter) Data for Electoral Analytics
This notebook uses snscrape to collect tweets without API access.
Useful for political sentiment, candidates, and campaign analysis.


In [5]:
# Diagnostico de entorno y carga de configuracion
import sys
import pandas as pd
import playwright

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

try:
    from importlib.metadata import version
    playwright_version = version("playwright")
except Exception:
    playwright_version = "No disponible"

print('Executable:', sys.executable)
print('Python version:', sys.version)
print('Playwright version:', playwright_version)
print('Pandas version:', pd.__version__)

Executable: c:\Users\diego\AppData\Local\Programs\Python\Python311\python.exe
Python version: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Playwright version: 1.58.0
Pandas version: 2.2.3


In [6]:
import sys
print(sys.executable)

c:\Users\diego\AppData\Local\Programs\Python\Python311\python.exe


## 1. Define Search Query

In [11]:
query = 'elecciones Colombia since:2022-01-01 until:2022-12-31 lang:es'
limit = 10

## 2. Scrape Tweets

In [13]:
import subprocess
import sys
from pathlib import Path

# 1. Definimos la salida
out_csv = Path("tweets_colombia.csv")

# 2. Construimos el comando con la estructura actual
cmd = [
    sys.executable,
    "playwright_scrape.py",
    "--query", query,
    "--headless",
    "--limit", str(limit),
]

print("Ejecutando:", " ".join(cmd))

# 3. Ejecutamos el scraper
subprocess.run(cmd, check=True)

Ejecutando: c:\Users\diego\AppData\Local\Programs\Python\Python311\python.exe playwright_scrape.py --query elecciones Colombia since:2022-01-01 until:2022-12-31 lang:es --headless --limit 10


CompletedProcess(args=['c:\\Users\\diego\\AppData\\Local\\Programs\\Python\\Python311\\python.exe', 'playwright_scrape.py', '--query', 'elecciones Colombia since:2022-01-01 until:2022-12-31 lang:es', '--headless', '--limit', '10'], returncode=0)

## 3. Load Dataset

In [ ]:
import pandas as pd

df = pd.read_csv(out_csv)
print(f"Dataset loaded: {out_csv} ({len(df)} rows)")
df.head()

## 4. Basic Analysis

In [ ]:
df['engagement'] = df['likes'] + df['retweets']
df.sort_values('engagement', ascending=False).head()

## 5. Sentiment Analysis
Pipeline de sentimiento y agregacion temporal usando el modelo cardiffnlp/twitter-xlm-roberta-base-sentiment.

In [14]:
from pathlib import Path
import subprocess
import sys
import pandas as pd

input_csv = out_csv
sentiment_csv = Path("tweets_colombia_sentiment.csv")
agg_csv = Path("tweets_colombia_agg.csv")
freq = "W"

cmd = [
    sys.executable,
    "sentiment_pipeline.py",
    "--input", str(input_csv),
    "--output", str(sentiment_csv),
    "--agg-output", str(agg_csv),
    "--freq", freq,
    "--batch-size", "32",
]
print("Ejecutando:", " ".join(cmd))
subprocess.run(cmd, check=True)

df_sent = pd.read_csv(sentiment_csv)
df_agg = pd.read_csv(agg_csv)

display(df_sent.head())
display(df_agg.head())

Ejecutando: c:\Users\diego\AppData\Local\Programs\Python\Python311\python.exe sentiment_pipeline.py --input tweets_colombia.csv --output tweets_colombia_sentiment.csv --agg-output tweets_colombia_agg.csv --freq W --batch-size 32


CalledProcessError: Command '['c:\\Users\\diego\\AppData\\Local\\Programs\\Python\\Python311\\python.exe', 'sentiment_pipeline.py', '--input', 'tweets_colombia.csv', '--output', 'tweets_colombia_sentiment.csv', '--agg-output', 'tweets_colombia_agg.csv', '--freq', 'W', '--batch-size', '32']' returned non-zero exit status 1.

## 6. Run sentiment pipeline
Ejecuta el pipeline de sentimiento de forma reproducible con CPU y carga los resultados.

In [ ]:
from pathlib import Path
import subprocess
import sys
import pandas as pd

input_csv = Path("tweets_colombia.csv")
sentiment_csv = Path("tweets_colombia_sentiment.csv")
agg_csv = Path("tweets_colombia_agg.csv")
freq = "W"

cmd = [
    sys.executable,
    "sentiment_pipeline.py",
    "--input", str(input_csv),
    "--output", str(sentiment_csv),
    "--agg-output", str(agg_csv),
    "--freq", freq,
    "--batch-size", "32",
    "--device", "cpu",
    "--log-level", "INFO",
]
print("Ejecutando:", " ".join(cmd))
subprocess.run(cmd, check=True)

df_sent = pd.read_csv(sentiment_csv)
df_agg = pd.read_csv(agg_csv)

display(df_sent.head())
display(df_agg.head())